In [130]:
import pandas as pd
import numpy as np

hourly = pd.read_csv("pedestrian_hourly.csv")

print("Shape:")
print(hourly.shape)

print("\nColumns:")
print(hourly.columns.tolist())

hourly.head()

Shape:
(1613800, 9)

Columns:
['ID', 'Location_ID', 'Sensing_Date', 'HourDay', 'Direction_1', 'Direction_2', 'Total_of_Directions', 'Sensor_Name', 'Location']


,ID,Location_ID,Sensing_Date,HourDay,Direction_1,Direction_2,Total_of_Directions,Sensor_Name,Location
0,49220260805,49,2026-08-05,2,6,11,17,Eli501_T,"-37.80730068, 144.95956055"
1,43020260805,43,2026-08-05,0,14,2,16,UM2_T,"-37.79844526, 144.96411782"
2,24120260805,24,2026-08-05,1,14,15,29,Col620_T,"-37.81887963, 144.95449198"
3,182120260805,182,2026-08-05,1,33,29,62,King163_T,"-37.81627451, 144.95550503"
4,30320260805,30,2026-08-05,3,0,1,1,Lon189_T,"-37.8112185, 144.96656806"


In [131]:
hourly.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1613800 entries, 0 to 1613799
Data columns (total 9 columns):
 #   Column               Non-Null Count    Dtype 
---  ------               --------------    ----- 
 0   ID                   1613800 non-null  int64 
 1   Location_ID          1613800 non-null  int64 
 2   Sensing_Date         1613800 non-null  object
 3   HourDay              1613800 non-null  int64 
 4   Direction_1          1613800 non-null  int64 
 5   Direction_2          1613800 non-null  int64 
 6   Total_of_Directions  1613800 non-null  int64 
 7   Sensor_Name          1588298 non-null  object
 8   Location             1588298 non-null  object
dtypes: int64(6), object(3)
memory usage: 110.8+ MB


In [132]:
hourly.isna().sum()

,0
ID,0
Location_ID,0
Sensing_Date,0
HourDay,0
Direction_1,0
Direction_2,0
Total_of_Directions,0
Sensor_Name,25502
Location,25502


In [133]:
hourly.duplicated().sum()

np.int64(0)

In [134]:
hourly.describe()

,ID,Location_ID,HourDay,Direction_1,Direction_2,Total_of_Directions
count,1.613800e+06,1.613800e+06,1.613800e+06,1.613800e+06,1.613800e+06,1.613800e+06
mean,4.924411e+11,7.582563e+01,1.174958e+01,1.956308e+02,1.982068e+02,3.938377e+02
std,5.441880e+11,5.458362e+01,6.793990e+00,3.057501e+02,3.105902e+02,5.859938e+02
min,1.020241e+09,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.662026e+10,3.100000e+01,6.000000e+00,1.800000e+01,1.800000e+01,3.800000e+01
50%,2.117203e+11,6.200000e+01,1.200000e+01,7.800000e+01,7.700000e+01,1.610000e+02
75%,7.512203e+11,1.300000e+02,1.800000e+01,2.340000e+02,2.390000e+02,4.900000e+02
max,2.092320e+12,2.090000e+02,2.300000e+01,1.009900e+04,1.108500e+04,1.111300e+04


In [135]:
q50 = hourly["Total_of_Directions"].quantile(0.50)
q75 = hourly["Total_of_Directions"].quantile(0.75)

print(q50)
print(q75)

161.0
490.0


In [136]:
def crowd_level(x):
    if x <= q50:
        return "Low"
    elif x <= q75:
        return "Medium"
    else:
        return "High"


hourly["Crowd_Level"] = hourly["Total_of_Directions"].apply(crowd_level)


hourly["Crowd_Level"].value_counts()

,count
Crowd_Level,
Low,808464
High,403042
Medium,402294


In [137]:
hourly["Sensing_Date"] = pd.to_datetime(hourly["Sensing_Date"])

hourly["Hour"] = hourly["Sensing_Date"].dt.hour

In [138]:
hourly.groupby("Hour")["Total_of_Directions"].mean()

,Total_of_Directions
Hour,
0,393.837666


In [139]:
hourly["Location"].head()

,Location
0,"-37.80730068, 144.95956055"
1,"-37.79844526, 144.96411782"
2,"-37.81887963, 144.95449198"
3,"-37.81627451, 144.95550503"
4,"-37.8112185, 144.96656806"


In [140]:
coord = hourly["Location"].str.split(",", expand=True)

coord.head()

,0,1
0,-37.80730068,144.95956055
1,-37.79844526,144.96411782
2,-37.81887963,144.95449198
3,-37.81627451,144.95550503
4,-37.8112185,144.96656806


In [141]:
hourly["Latitude"] = pd.to_numeric(coord[0])

hourly["Longitude"] = pd.to_numeric(coord[1])

In [142]:
hourly[
    [
        "Location",
        "Latitude",
        "Longitude"
    ]
].head()

,Location,Latitude,Longitude
0,"-37.80730068, 144.95956055",-37.807301,144.959561
1,"-37.79844526, 144.96411782",-37.798445,144.964118
2,"-37.81887963, 144.95449198",-37.818880,144.954492
3,"-37.81627451, 144.95550503",-37.816275,144.955505
4,"-37.8112185, 144.96656806",-37.811219,144.966568


In [143]:
print("Missing Latitude:", hourly["Latitude"].isna().sum())
print("Missing Longitude:", hourly["Longitude"].isna().sum())

Missing Latitude: 25502
Missing Longitude: 25502


In [144]:
print("Total rows:", len(hourly))

print(
    "Missing coordinates:",
    hourly["Latitude"].isna().sum()
)

print(
    "Missing percentage:",
    hourly["Latitude"].isna().mean()*100
)

Total rows: 1613800
Missing coordinates: 25502
Missing percentage: 1.580245383566737


In [145]:
hourly_clean = hourly.dropna(
    subset=[
        "Latitude",
        "Longitude"
    ]
).copy()

In [146]:
print(
    hourly_clean.shape
)

print(
    hourly_clean[
        ["Latitude","Longitude"]
    ].isna().sum()
)

(1588298, 13)
Latitude     0
Longitude    0
dtype: int64


In [147]:
hourly_clean["Sensing_Date"] = pd.to_datetime(
    hourly_clean["Sensing_Date"]
)

hourly_clean = hourly_clean.sort_values(
    [
        "Location_ID",
        "Sensing_Date",
        "HourDay"
    ]
)

hourly_clean.head()

,ID,Location_ID,Sensing_Date,HourDay,Direction_1,Direction_2,Total_of_Directions,Sensor_Name,Location,Crowd_Level,Hour,Latitude,Longitude
1613472,1020240806,1,2024-08-06,0,10,12,22,Bou292_T,"-37.81349441, 144.96515323",Low,0,-37.813494,144.965153
1612827,1120240806,1,2024-08-06,1,7,9,16,Bou292_T,"-37.81349441, 144.96515323",Low,0,-37.813494,144.965153
1612195,1220240806,1,2024-08-06,2,5,6,11,Bou292_T,"-37.81349441, 144.96515323",Low,0,-37.813494,144.965153
1613720,1320240806,1,2024-08-06,3,3,0,3,Bou292_T,"-37.81349441, 144.96515323",Low,0,-37.813494,144.965153
1612343,1420240806,1,2024-08-06,4,1,9,10,Bou292_T,"-37.81349441, 144.96515323",Low,0,-37.813494,144.965153


In [148]:
hourly_clean["Next_Hour_Crowd"] = (
    hourly_clean
    .groupby("Location_ID")["Total_of_Directions"]
    .shift(-1)
)

In [149]:
hourly_clean[
    [
        "Location_ID",
        "HourDay",
        "Total_of_Directions",
        "Next_Hour_Crowd"
    ]
].head(10)

,Location_ID,HourDay,Total_of_Directions,Next_Hour_Crowd
1613472,1,0,22,16.0
1612827,1,1,16,11.0
1612195,1,2,11,3.0
1613720,1,3,3,10.0
1612343,1,4,10,13.0
1612782,1,5,13,74.0
1612299,1,6,74,220.0
1612576,1,7,220,494.0
1613495,1,8,494,600.0
1612257,1,9,600,864.0


In [150]:
hourly_clean = hourly_clean.dropna(
    subset=["Next_Hour_Crowd"]
)

In [151]:
future_q50 = hourly_clean["Next_Hour_Crowd"].quantile(0.5)

future_q75 = hourly_clean["Next_Hour_Crowd"].quantile(0.75)


def future_alert(x):
    if x <= future_q50:
        return "Low"
    elif x <= future_q75:
        return "Medium"
    else:
        return "High"


hourly_clean["Alert_Level"] = (
    hourly_clean["Next_Hour_Crowd"]
    .apply(future_alert)
)


hourly_clean["Alert_Level"].value_counts()

,count
Alert_Level,
Low,794850
High,396797
Medium,396551


In [152]:
minute = pd.read_csv("pedestrian_minute.csv")

minute.head()

,Location_ID,Sensing_DateTime,Sensing_Date,Sensing_Time,Direction_1,Direction_2,Total_of_Directions
0,3,2026-08-06T15:04:00+10:00,2026-08-06,15:04,15,21,36
1,69,2026-08-06T15:04:00+10:00,2026-08-06,15:04,5,10,15
2,10,2026-08-06T15:04:00+10:00,2026-08-06,15:04,6,3,9
3,11,2026-08-06T15:04:00+10:00,2026-08-06,15:04,1,0,1
4,18,2026-08-06T15:04:00+10:00,2026-08-06,15:04,3,2,5


In [153]:
minute.info()
minute.columns.tolist()
minute.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197798 entries, 0 to 197797
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   Location_ID          197798 non-null  int64 
 1   Sensing_DateTime     197798 non-null  object
 2   Sensing_Date         197798 non-null  object
 3   Sensing_Time         197798 non-null  object
 4   Direction_1          197798 non-null  int64 
 5   Direction_2          197798 non-null  int64 
 6   Total_of_Directions  197798 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 10.6+ MB


,0
Location_ID,0
Sensing_DateTime,0
Sensing_Date,0
Sensing_Time,0
Direction_1,0
Direction_2,0
Total_of_Directions,0


In [154]:
minute["Total_of_Directions"].describe()

,Total_of_Directions
count,197798.000000
mean,13.758658
std,22.396004
min,0.000000
25%,2.000000
50%,6.000000
75%,16.000000
max,533.000000


In [155]:

minute.duplicated().sum()

np.int64(0)

In [156]:
minute["Sensing_DateTime"] = pd.to_datetime(
    minute["Sensing_DateTime"]
)

In [157]:
minute["Sensing_DateTime"].min(), minute["Sensing_DateTime"].max()

(Timestamp('2026-08-02 23:55:00+1000', tz='UTC+10:00'),
 Timestamp('2026-08-06 15:04:00+1000', tz='UTC+10:00'))

In [158]:
minute["Location_ID"].nunique()

99

In [159]:
minute.groupby("Location_ID").size().describe()

,0
count,99.000000
mean,1997.959596
std,1354.859115
min,36.000000
25%,856.500000
50%,1019.000000
75%,3147.000000
max,4671.000000


In [160]:
minute = minute.sort_values(
    ["Location_ID", "Sensing_DateTime"]
)

minute["Time_Diff"] = (
    minute.groupby("Location_ID")["Sensing_DateTime"]
    .diff()
)

minute["Time_Diff"].value_counts().head()

,count
Time_Diff,
0 days 00:01:00,136810
0 days 00:05:00,38022
0 days 00:02:00,10336
0 days 00:03:00,3648
0 days 00:10:00,1782


In [161]:
minute[
    minute["Total_of_Directions"] > 200
].head()

,Location_ID,Sensing_DateTime,Sensing_Date,Sensing_Time,Direction_1,Direction_2,Total_of_Directions,Time_Diff
63607,5,2026-08-05 12:50:00+10:00,2026-08-05,12:50,126,76,202,0 days 00:05:00
52385,5,2026-08-05 16:15:00+10:00,2026-08-05,16:15,284,44,328,0 days 00:05:00
52076,5,2026-08-05 16:20:00+10:00,2026-08-05,16:20,195,50,245,0 days 00:05:00
104759,47,2026-08-04 17:05:00+10:00,2026-08-04,17:05,126,77,203,0 days 00:05:00
104439,47,2026-08-04 17:10:00+10:00,2026-08-04,17:10,129,86,215,0 days 00:05:00


In [162]:
minute["Crowd_Change"] = (
    minute.groupby("Location_ID")
    ["Total_of_Directions"]
    .diff()
)

minute.head()

,Location_ID,Sensing_DateTime,Sensing_Date,Sensing_Time,Direction_1,Direction_2,Total_of_Directions,Time_Diff,Crowd_Change
197671,1,2026-08-03 00:00:00+10:00,2026-08-03,00:00,4,0,4,NaT,NaN
197550,1,2026-08-03 00:05:00+10:00,2026-08-03,00:05,1,0,1,0 days 00:05:00,-3.0
197517,1,2026-08-03 00:06:00+10:00,2026-08-03,00:06,0,1,1,0 days 00:01:00,0.0
197507,1,2026-08-03 00:07:00+10:00,2026-08-03,00:07,0,2,2,0 days 00:01:00,1.0
197482,1,2026-08-03 00:08:00+10:00,2026-08-03,00:08,0,1,1,0 days 00:01:00,-1.0


In [163]:
q75 = minute["Crowd_Change"].quantile(0.75)

print(q75)

3.0


In [164]:
q75 = minute["Crowd_Change"].quantile(0.75)

def trend_alert(x):
    if x <= 0:
        return "Stable"
    elif x <= q75:
        return "Increasing"
    else:
        return "Rapid Increase"


minute["Trend_Alert"] = (
    minute["Crowd_Change"]
    .apply(trend_alert)
)

minute["Trend_Alert"].value_counts()

,count
Trend_Alert,
Stable,113524
Increasing,42386
Rapid Increase,41888


In [165]:
minute["Crowd_Change"] = (
    minute.groupby("Location_ID")
    ["Total_of_Directions"]
    .diff()
)

minute["Crowd_Change"] = minute["Crowd_Change"].fillna(0)

In [166]:
minute_alert = minute[
[
"Location_ID",
"Sensing_DateTime",
"Trend_Alert"
]
]

In [167]:
minute_alert_output = minute[
[
    "Location_ID",
    "Sensing_DateTime",
    "Total_of_Directions",
    "Crowd_Change",
    "Trend_Alert"
]
].copy()


minute_alert_output.to_csv(
    "minute_alert_prediction.csv",
    index=False
)

In [168]:
hourly_alert_output = hourly_clean[
[
    "Location_ID",
    "Sensing_Date",
    "HourDay",
    "Latitude",
    "Longitude",
    "Total_of_Directions",
    "Next_Hour_Crowd",
    "Alert_Level"
]
].copy()


hourly_alert_output.to_csv(
    "hourly_alert_prediction.csv",
    index=False
)